# Research Template: Batch Growth Calculations

Use this template to process a CSV of growth measurements and output SDS, centiles, and quality flags. Copy (do not edit) this template when starting a new analysis to preserve provenance.

Sections:
1. Configuration & Environment
2. Load Input Data
3. Data Validation & Cleaning
4. Derive Ages & (Optional) Correct for Prematurity
5. Compute SDS & Centiles
6. Add Quality / Plausibility Flags
7. Summaries & Visual Quality Checks
8. Export Augmented Dataset

> ALWAYS de‑identify: no names, NHS numbers, addresses, exact birth dates if not required. Consider offsetting all dates by a fixed random number of days per subject if sharing.


## 1. Configuration & Environment
Adjust paths & parameters here.
- INPUT_CSV: path to your de‑identified dataset
- OUTPUT_CSV: destination for augmented results
- REFERENCE_SET: choose 'uk_who' (default) or see other reference selector functions


In [ ]:
from pathlib import Path
import sys, platform
import pandas as pd
import rcpchgrowth
from datetime import date

INPUT_CSV = Path('YOUR_INPUT_FILE.csv')  # <-- change
OUTPUT_CSV = Path('augmented_results.csv')
REFERENCE_SET = 'uk-who'  # placeholder for switching logic

print({'python': sys.version.split()[0], 'platform': platform.platform(), 'rcpchgrowth': getattr(rcpchgrowth,'__version__','unknown'), 'pandas': pd.__version__})
print('Input file exists?', INPUT_CSV.exists())

## 2. Load Input Data
Expected columns (rename your fields if needed):
- id (string / pseudonym)
- sex ("M"/"F")
- dob (YYYY-MM-DD)
- measurement_date (YYYY-MM-DD)
- weight_kg (optional if computing weight SDS)
- height_cm (optional if computing height SDS)

Add any BMI or head circumference columns similarly.


In [ ]:
if INPUT_CSV.exists():
    df = pd.read_csv(INPUT_CSV, parse_dates=['dob','measurement_date'])
else:
    # Create demo frame (remove in production)
    from datetime import date
    df = pd.DataFrame([
        {'id':'A','sex':'F','dob':'2022-06-15','measurement_date':'2024-02-01','weight_kg':12.3,'height_cm':87.2},
        {'id':'B','sex':'M','dob':'2021-11-10','measurement_date':'2024-02-01','weight_kg':15.0,'height_cm':93.4},
    ])
    df['dob'] = pd.to_datetime(df['dob'])
    df['measurement_date'] = pd.to_datetime(df['measurement_date'])

print(df.head())
print(df.dtypes)

## 3. Data Validation & Cleaning
Lightweight checks: required columns, duplicate ids + dates, missing values.


In [ ]:
required = {'id','sex','dob','measurement_date'}
missing_cols = required - set(df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

# Basic duplicates check
dup_mask = df.duplicated(subset=['id','measurement_date'])
if dup_mask.any():
    print('Warning: duplicate id+measurement_date rows found')
    display(df[dup_mask])

# Simple missing values report
na_counts = df.isna().sum()
print('Missing values per column:\n', na_counts)

## 4. Derive Ages
We'll compute chronological decimal age for each measurement. (Add corrected age logic if you have gestational age data.)


In [ ]:
from rcpchgrowth import chronological_decimal_age

df['age_decimal_years'] = [chronological_decimal_age(d.date(), m.date()) for d,m in zip(df['dob'], df['measurement_date'])]
df[['id','age_decimal_years']].head()

## 5. Compute SDS & Centiles
We'll calculate weight and height SDS / centiles where data is present.


In [ ]:
from rcpchgrowth import sds_for_measurement, centile, select_reference_data_for_uk_who_chart

def compute_sds(row, measurement_name, value_col):
    val = row.get(value_col)
    if pd.isna(val):
        return pd.Series({f'{measurement_name}_sds': pd.NA, f'{measurement_name}_centile': pd.NA})
    ref = select_reference_data_for_uk_who_chart(measurement_name, row.sex, row.age_decimal_years)
    sds_val = sds_for_measurement(measurement_name, val, ref)
    return pd.Series({f'{measurement_name}_sds': sds_val, f'{measurement_name}_centile': centile(sds_val)})

for m, col in [('weight','weight_kg'), ('height','height_cm')]:
    res = df.apply(lambda r: compute_sds(r, m, col), axis=1)
    df = pd.concat([df, res], axis=1)

df.head()

## 6. Quality / Plausibility Flags
Simple biologically implausible value (BIV) flags using SDS thresholds (+/- 6 as placeholder; adjust to policy) and missingness.


In [ ]:
def flag_biv(sds):
    try:
        return abs(sds) > 6
    except TypeError:
        return pd.NA

for m in ['weight','height']:
    df[f'{m}_biv_flag'] = df[f'{m}_sds'].apply(flag_biv)

# Missingness summary
metrics = ['weight_kg','height_cm']
missing_summary = {m: df[m].isna().mean() for m in metrics}
print('Missingness fraction:', missing_summary)
df.head()

## 7. Summaries & Visual Quality Checks
Distributions & simple trends help spot outliers.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,2, figsize=(8,3))
axes[0].hist(df['weight_sds'].dropna(), bins=10, color='skyblue')
axes[0].set_title('Weight SDS')
axes[1].hist(df['height_sds'].dropna(), bins=10, color='salmon')
axes[1].set_title('Height SDS')
plt.tight_layout()
plt.show()

df[['weight_sds','height_sds']].describe()

## 8. Export Augmented Dataset
Write out enriched data. DO NOT commit sensitive outputs.


In [ ]:
# Uncomment to export
# df.to_csv(OUTPUT_CSV, index=False)
print('Augmented dataset ready. Uncomment export line to save.')